In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [3]:

# CSV'yi oku
df = pd.read_csv('outputs/logs/cls_fold0.csv')

print("=" * 80)
print("EĞİTİM SONUÇLARI DETAYLI ANALİZİ")
print("=" * 80)

# 1. Genel Performans Özeti
print("\n1. GENEL PERFORMANS:")
print(f"   • Başlangıç mAP: {df.iloc[0]['mAP']:.4f}")
print(f"   • Son mAP: {df.iloc[-1]['mAP']:.4f}")
print(f"   • Maksimum mAP: {df['mAP'].max():.4f} (Epoch {df['mAP'].idxmax()})")
print(f"   • mAP iyileştirme: +{(df['mAP'].max() - df.iloc[0]['mAP']):.4f}")
print(f"\n   • Başlangıç macroF1: {df.iloc[0]['macroF1']:.4f}")
print(f"   • Son macroF1: {df.iloc[-1]['macroF1']:.4f}")
print(f"   • Maksimum macroF1: {df['macroF1'].max():.4f} (Epoch {df['macroF1'].idxmax()})")


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/logs/cls_fold0.csv'

In [ ]:


# 2. Sınıf Performansı Analizi
print("\n2. SINIF BAZLI PERFORMANS (Final - Epoch 49):")
classes = ['acute_cholecystitis', 'kidney_ureter_stone', 'acute_pancreatitis', 
           'aortic_aneurysm_dissection', 'acute_appendicitis', 'acute_diverticulitis']

final_row = df.iloc[-1]
performance_data = []

for cls in classes:
    ap = final_row[f'AP/{cls}']
    f1 = final_row[f'F1/{cls}']
    performance_data.append({
        'Class': cls,
        'AP': ap,
        'F1': f1,
        'Avg': (ap + f1) / 2
    })

perf_df = pd.DataFrame(performance_data).sort_values('AP', ascending=False)
print(perf_df.to_string(index=False))

# 3. Sınıf Performans Kategorileri
print("\n3. SINIF KATEGORİLERİ:")
print("   ⭐ Mükemmel (AP > 0.95, F1 > 0.93):")
for _, row in perf_df.iterrows():
    if row['AP'] > 0.95 and row['F1'] > 0.93:
        print(f"      • {row['Class']}: AP={row['AP']:.4f}, F1={row['F1']:.4f}")

print("\n   ✅ İyi (AP > 0.85, F1 > 0.80):")
for _, row in perf_df.iterrows():
    if 0.85 < row['AP'] <= 0.95 or 0.80 < row['F1'] <= 0.93:
        print(f"      • {row['Class']}: AP={row['AP']:.4f}, F1={row['F1']:.4f}")

print("\n   ⚠️ Orta (AP > 0.60, F1 > 0.50):")
for _, row in perf_df.iterrows():
    if 0.60 < row['AP'] <= 0.85 or 0.50 < row['F1'] <= 0.80:
        print(f"      • {row['Class']}: AP={row['AP']:.4f}, F1={row['F1']:.4f}")

print("\n   ❌ Zayıf (AP < 0.60 veya F1 < 0.50):")
for _, row in perf_df.iterrows():
    if row['AP'] < 0.60 or row['F1'] < 0.50:
        print(f"      • {row['Class']}: AP={row['AP']:.4f}, F1={row['F1']:.4f}")

# 4. Overfitting Analizi
print("\n4. OVERFITTING ANALİZİ:")
print(f"   • Train Loss (son): {df.iloc[-1]['train_loss']:.2e}")
print(f"   • Train Loss Trend: Başlangıç {df.iloc[0]['train_loss']:.6f} → Son {df.iloc[-1]['train_loss']:.2e}")
print(f"   • mAP Trend: Başlangıç {df.iloc[0]['mAP']:.4f} → Pik {df['mAP'].max():.4f} → Son {df.iloc[-1]['mAP']:.4f}")

if df.iloc[-1]['train_loss'] < 0.0001:
    print("   ⚠️ Loss çok düşük - hafif overfitting riski")
    
if df.iloc[-1]['mAP'] < df['mAP'].max() - 0.01:
    print(f"   ⚠️ mAP düşüş: Pik {df['mAP'].max():.4f} → Son {df.iloc[-1]['mAP']:.4f}")

# 5. Convergence Analizi
print("\n5. CONVERGENCE ANALİZİ:")
last_10_map = df.iloc[-10:]['mAP'].std()
print(f"   • Son 10 epoch mAP değişimi: {last_10_map:.6f} (çok küçük = converged)")
if last_10_map < 0.001:
    print("   ✅ Model converged")
    
# 6. Sorunlu Sınıflar
print("\n6. SORUNLU SINIFLAR (İyileştirme Gerekli):")
weak = perf_df[perf_df['AP'] < 0.70]
for _, row in weak.iterrows():
    gap = row['AP'] - row['F1']
    print(f"   • {row['Class']}: AP={row['AP']:.4f}, F1={row['F1']:.4f}")
    print(f"     └─ AP-F1 Farkı: {gap:.4f} (Precision vs Recall dengesizliği)")